# MedCity Analytics — Healthcare Billing EDA Deep Dive
**The Darko Method 2026 | Module 06**

Answering four CFO-level questions:
- **Q1** — Which `insurance_type` has the lowest `recovery_rate_pct`? Highest `outstanding_balance`?
- **Q2** — Does `age_from_dob` correlate with `lifetime_billed` or `bill_collection_rate`?
- **Q3** — Is `insurance_type` statistically independent of billing outcome? (Chi-square test)
- **Q4** — Which patients are low-recovery outliers at risk of write-off?

In [ ]:
import sys, pathlib
root = pathlib.Path().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from scipy import stats

from src.eda_engine               import HealthcareEDAEngine
from src.patient_segment_profiler import PatientSegmentProfiler
from src.anomaly_detector         import AnomalyDetector

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

engine = HealthcareEDAEngine()
(
    engine
    .load().profile().group_analysis().correlation()
    .chi_square_test('insurance_type', 'bill_collection_rate')
    .recovery_by_group('insurance_type')
    .age_correlation('lifetime_billed')
    .age_correlation('bill_collection_rate')
)
df = engine.df

profiler = PatientSegmentProfiler(df)
profiler.profile_all()

detector = AnomalyDetector(df)
detector.run(columns=['lifetime_billed', 'outstanding_balance', 'bill_collection_rate'])
detector.flag_low_recovery()

print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')
df.head(3)

---
## Chart 1 — Recovery Rate by Insurance Type
**Q1:** Which insurance type recovers the most? Which the least?

In [ ]:
rows = engine.results['recovery_by_group']['insurance_type']
rec_df = pd.DataFrame(rows).sort_values('recovery_rate_pct', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: recovery_rate_pct bar
colors = sns.color_palette('Blues_r', len(rec_df))
bars = axes[0].barh(rec_df['insurance_type'], rec_df['recovery_rate_pct'],
                    color=colors, edgecolor='white')
for bar, val in zip(bars, rec_df['recovery_rate_pct']):
    axes[0].text(val + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}%', va='center', fontsize=10)
axes[0].set_xlim(93, 95)
axes[0].set_xlabel('Mean Recovery Rate (%)', fontsize=12)
axes[0].set_title('Q1 — Recovery Rate by Insurance Type', fontsize=12, fontweight='bold')

# Right: bill_collection_rate box plot
order = rec_df['insurance_type'].tolist()
sns.boxplot(data=df, x='insurance_type', y='bill_collection_rate',
            order=order[::-1], palette='Blues', ax=axes[1])
axes[1].set_xlabel('Insurance Type', fontsize=12)
axes[1].set_ylabel('Bill Collection Rate (%)', fontsize=12)
axes[1].set_title('Q1 — Collection Rate Distribution by Insurance Type',
                  fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print('Recovery rate ranking (lowest first):')
for r in rows:
    print(f"  Rank {r['rank']}: {r['insurance_type']:<12}  recovery={r['recovery_rate_pct']:.3f}%  collection={r['bill_collection_rate']:.3f}%")

---
## Chart 2 — Lifetime Billed Distribution by Insurance Type
**Q1:** Which type carries the highest outstanding balance and billed amount?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
order = (
    df.groupby('insurance_type')['lifetime_billed']
    .median().sort_values(ascending=False).index
)
sns.boxplot(data=df, x='insurance_type', y='lifetime_billed',
            order=order, palette='Reds_r', ax=ax)
ax.axhline(df['lifetime_billed'].mean(), color='#2ecc71', linestyle='--', linewidth=1.5,
           label=f'Overall mean ${df["lifetime_billed"].mean():,.0f}')
ax.set_xlabel('Insurance Type', fontsize=12)
ax.set_ylabel('Lifetime Billed (USD)', fontsize=12)
ax.set_title('Q1 — Lifetime Billed by Insurance Type', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('Mean lifetime_billed by insurance type:')
print(df.groupby('insurance_type')['lifetime_billed'].mean().sort_values(ascending=False).round(2).to_string())

---
## Chart 3 — Age vs Billing Outcomes
**Q2:** Does patient age correlate with lifetime_billed or bill_collection_rate?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = df.sample(min(2000, len(df)), random_state=42)
palette_ins = dict(zip(df['insurance_type'].unique(),
                       sns.color_palette('Set2', df['insurance_type'].nunique())))

for ins, grp in sample.groupby('insurance_type'):
    axes[0].scatter(grp['age_from_dob'], grp['lifetime_billed'],
                    alpha=0.3, s=10, color=palette_ins.get(ins, '#95a5a6'), label=ins)
axes[0].set_xlabel('Patient Age', fontsize=12)
axes[0].set_ylabel('Lifetime Billed (USD)', fontsize=12)
axes[0].set_title('Q2 — Age vs Lifetime Billed', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8, markerscale=2)

for ins, grp in sample.groupby('insurance_type'):
    axes[1].scatter(grp['age_from_dob'], grp['bill_collection_rate'],
                    alpha=0.3, s=10, color=palette_ins.get(ins, '#95a5a6'), label=ins)
axes[1].set_xlabel('Patient Age', fontsize=12)
axes[1].set_ylabel('Bill Collection Rate (%)', fontsize=12)
axes[1].set_title('Q2 — Age vs Bill Collection Rate', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

for col in ['lifetime_billed', 'bill_collection_rate']:
    res = engine.results['age_correlation'].get(col, {})
    if res:
        print(f'age_from_dob ~ {col}: r={res["r"]:.4f}  p={res["p_value"]:.4f}  '
              f'[{"SIGNIFICANT" if res["significant"] else "not significant"}]')

---
## Chart 4 — Chi-Square Test: Insurance Type vs Billing Outcome
**Q3:** Is insurance_type statistically independent of bill_collection_rate?

In [ ]:
# Build bucketed contingency table for visualization
median_val = df['bill_collection_rate'].median()
df_plot = df.copy()
df_plot['collection_bucket'] = df_plot['bill_collection_rate'].apply(
    lambda x: 'High' if x >= median_val else 'Low'
)

ct = pd.crosstab(df_plot['insurance_type'], df_plot['collection_bucket'],
                 normalize='index') * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: stacked bar of High/Low proportions
ct.plot(kind='bar', stacked=True, ax=axes[0],
        color=['#e74c3c', '#3498db'], edgecolor='white', width=0.6)
axes[0].set_xlabel('Insurance Type', fontsize=12)
axes[0].set_ylabel('Share of Patients (%)', fontsize=12)
axes[0].set_title('Q3 — Collection Rate (High/Low) by Insurance Type',
                  fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(title='Collection Rate', fontsize=10)

# Right: violin plot of raw bill_collection_rate
order_ins = df.groupby('insurance_type')['bill_collection_rate'].mean().sort_values().index
sns.violinplot(data=df, x='insurance_type', y='bill_collection_rate',
               order=order_ins, palette='Set2', ax=axes[1], cut=0)
axes[1].set_xlabel('Insurance Type', fontsize=12)
axes[1].set_ylabel('Bill Collection Rate (%)', fontsize=12)
axes[1].set_title('Q3 — Collection Rate Distribution', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

# Print chi-square result
chi2_res = engine.results['chi_square'].get('insurance_type ~ bill_collection_rate', {})
if chi2_res:
    print(f'Chi-square test: chi2={chi2_res["chi2"]:.3f}  p={chi2_res["p_value"]:.4f}  dof={chi2_res["dof"]}')
    print(f'Result: {chi2_res["verdict"]}')

---
## Chart 5 — Low-Recovery Outliers
**Q4:** Which patients have statistically unusual low collection rates?

In [ ]:
lr = detector.results.get('low_recovery', {})
fence = lr.get('lower_fence', None)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram with fence line
sns.histplot(df['bill_collection_rate'], bins=40, kde=True,
             color='#3498db', ax=axes[0])
if fence is not None and fence > 0:
    axes[0].axvline(fence, color='#e74c3c', linewidth=2, linestyle='--',
                    label=f'IQR lower fence ({fence:.1f}%)')
    axes[0].legend(fontsize=10)
axes[0].set_xlabel('Bill Collection Rate (%)', fontsize=12)
axes[0].set_ylabel('Number of Patients', fontsize=12)
axes[0].set_title('Q4 — Bill Collection Rate Distribution', fontsize=12, fontweight='bold')

# Right: low-recovery patients breakdown by insurance type
if fence is not None:
    low_patients = df[df['bill_collection_rate'] < fence] if fence > 0 else df[df['bill_collection_rate'] < df['bill_collection_rate'].quantile(0.25)]
    low_counts = low_patients['insurance_type'].value_counts().sort_values(ascending=True)
    palette = sns.color_palette('Reds_r', len(low_counts))
    bars = axes[1].barh(low_counts.index, low_counts.values, color=palette, edgecolor='white')
    for bar, val in zip(bars, low_counts.values):
        axes[1].text(val + 0.3, bar.get_y() + bar.get_height()/2,
                     f'{val}', va='center', fontsize=10)
    axes[1].set_xlabel('Number of Low-Recovery Patients', fontsize=12)
    axes[1].set_title('Q4 — Low-Recovery Patients by Insurance Type',
                      fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

if lr:
    print(f'IQR lower fence: {lr["lower_fence"]:.2f}%')
    print(f'Flagged patients: {lr["flagged_count"]:,} ({lr["flagged_pct"]}% of all patients)')
    print(f'Mean collection rate among flagged: {lr["mean_rate"]:.2f}%')
    print(f'Full list saved to: reports/anomalies.csv')

---
## Summary of Findings

| Question | Finding |
|---|---|
| Q1 — Recovery by insurance | Medicare has lowest recovery (93.94%); Medicaid carries highest billed amount; spread is only ~0.44 pp across all types |
| Q1 — Outstanding balance | outstanding_balance = 0 for all patients in this dataset — fully recovered or written off |
| Q2 — Age and billing | age_from_dob has negligible correlation with both lifetime_billed (r=0.013) and bill_collection_rate (r=−0.003) |
| Q3 — Chi-square test | SIGNIFICANT (p=0.039) — insurance type is NOT independent of billing outcome; predicts whether bills get collected |
| Q4 — Low-recovery outliers | 384 patients (7.9%) below IQR fence of 76.8%; mean collection rate 59% — priority for billing follow-up |

**Outputs generated:**
- `reports/analysis_report.txt` — full text EDA report
- `reports/segment_profile.csv` — billing profile by insurance type
- `reports/anomalies.csv` — confirmed statistical outliers